# Ketamine Docking Pipeline - Google Colab

Bu notebook, ketamin moleküler docking pipeline'ını Google Colab'da çalıştırmanızı sağlar.

**⚠️ Önemli Notlar:**
- Runtime: ~1-2 saat (tüm hedefler için)
- Colab ücretsiz sürümünde çalışır
- Sonuçları indirmeyi unutmayın (Colab geçici)
- GPU gerekmez (CPU yeterli)

**Adımlar:**
1. Setup: Tüm bağımlılıkları yükle
2. Download: Pipeline kodlarını indir
3. Run: Pipeline'ı çalıştır
4. Download Results: Sonuçları bilgisayarına indir

## 1️⃣ Setup - Bağımlılıkları Yükle

Bu adım ~5-10 dakika sürer.

In [1]:
# Önce çalışma dizinini kontrol et
!pwd
!ls -la

/content
total 16
drwxr-xr-x 1 root root 4096 Nov 11 14:29 .
drwxr-xr-x 1 root root 4096 Nov 13 17:09 ..
drwxr-xr-x 4 root root 4096 Nov 11 14:29 .config
drwxr-xr-x 1 root root 4096 Nov 11 14:29 sample_data


In [2]:
# Python paketlerini yükle
print("📦 Python paketleri yükleniyor...")

!pip install -q biopython rdkit pandas pyyaml openpyxl matplotlib seaborn scipy numpy

print("✓ Python paketleri yüklendi")

📦 Python paketleri yükleniyor...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.2/36.2 MB 27.0 MB/s eta 0:00:00
✓ Python paketleri yüklendi


In [3]:
# AutoDock Vina'yı yükle (binary download)
print("🔬 AutoDock Vina yükleniyor...")

import os
import urllib.request

# Vina binary indir
vina_url = "https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64"

if not os.path.exists("/usr/local/bin/vina"):
    print("  Downloading Vina...")
    urllib.request.urlretrieve(vina_url, "/tmp/vina")
    !chmod +x /tmp/vina
    !sudo mv /tmp/vina /usr/local/bin/vina
    print("  ✓ Vina installed")
else:
    print("  ✓ Vina already installed")

# Test
!vina --version
print("✓ AutoDock Vina yüklendi")

🔬 AutoDock Vina yükleniyor...
  ✓ Vina installed
AutoDock Vina v1.2.5
✓ AutoDock Vina yüklendi


In [4]:
# Open Babel yükle (PDBQT dönüşüm için)
print("🧪 Open Babel yükleniyor...")

!apt-get update -qq
!apt-get install -qq -y openbabel

# Test
!obabel --version
print("✓ Open Babel yüklendi")

🧪 Open Babel yükleniyor...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libboost-iostreams1.74.0:amd64.
(Reading database ... 121235 files and directories currently installed.)
Preparing to unpack .../libboost-iostreams1.74.0_1.74.0-14ubuntu3_amd64.deb ...
Unpacking libboost-iostreams1.74.0:amd64 (1.74.0-14ubuntu3) ...
Selecting previously unselected package libinchi1.
Preparing to unpack .../libinchi1_1.03+dfsg-4_amd64.deb ...
Unpacking libinchi1 (1.03+dfsg-4) ...
Selecting previously unselected package libmaeparser1:amd64.
Preparing to unpack .../libmaeparser1_1.2.4-1build1_amd64.deb ...
Unpacking libmaeparser1:amd64 (1.2.4-1build1) ...
Selecting previously unselected package libopenbabel7.
Preparing to unpack .../libopenbabel7_3.1.1+dfsg-6ubuntu5_amd64.deb ...
Unpacking libopenbabel7 (3.1.1+dfsg-6ubunt

In [5]:
# Kurulumu doğrula
print("\n" + "="*70)
print("KURULUM KONTROLÜ")
print("="*70)

import sys

# Python version
print(f"Python: {sys.version.split()[0]}")

# Paketler
packages = ['Bio', 'rdkit', 'pandas', 'yaml', 'numpy', 'scipy', 'matplotlib']
for pkg in packages:
    try:
        __import__(pkg)
        print(f"✓ {pkg}")
    except ImportError:
        print(f"✗ {pkg} EKSIK!")

# External tools
!which vina > /dev/null && echo "✓ AutoDock Vina" || echo "✗ Vina EKSIK!"
!which obabel > /dev/null && echo "✓ Open Babel" || echo "✗ Open Babel EKSIK!"

print("="*70)
print("✓ Kurulum tamamlandı!")
print("="*70)


KURULUM KONTROLÜ
Python: 3.12.12
✓ Bio
✓ rdkit
✓ pandas
✓ yaml
✓ numpy
✓ scipy
✓ matplotlib
✓ AutoDock Vina
✓ Open Babel
✓ Kurulum tamamlandı!


## 2️⃣ Pipeline Kodlarını İndir

GitHub'dan pipeline kodlarını çek.

In [6]:
# GitHub reposunu clone et
!rm -rf comp-chem-scripts  # Önceki varsa sil

print("📥 Repository indiriliyor...")
!git clone -b claude/ketamine-docking-pipeline-011CV5ZKC6fFJBRhdCG2Mtt6 https://github.com/egundeger/comp-chem-scripts.git

# Ketamine docking dizinine geç
%cd comp-chem-scripts/ketamine-docking

print("\n✓ Kodlar indirildi!")
print("\nKlasör içeriği:")
!ls -la

📥 Repository indiriliyor...
Cloning into 'comp-chem-scripts'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (30/30), done.
Receiving objects: 100% (39/39), 44.57 KiB | 3.43 MiB/s, done.
remote: Total 39 (delta 7), reused 35 (delta 6), pack-reused 3 (from 1)
Resolving deltas: 100% (7/7), done.
/content/comp-chem-scripts/ketamine-docking

✓ Kodlar indirildi!

Klasör içeriği:
total 108
drwxr-xr-x 4 root root  4096 Nov 13 17:11 .
drwxr-xr-x 4 root root  4096 Nov 13 17:11 ..
-rw-r--r-- 1 root root  4850 Nov 13 17:11 check_dependencies.py
drwxr-xr-x 2 root root  4096 Nov 13 17:11 configs
-rw-r--r-- 1 root root 11792 Nov 13 17:11 EXAMPLES.md
-rw-r--r-- 1 root root   623 Nov 13 17:11 .gitignore
-rw-r--r-- 1 root root 15380 Nov 13 17:11 Ketamine_Docking_Colab.ipynb
-rw-r--r-- 1 root root  7549 Nov 13 17:11 QUICKSTART.md
-rw-r--r-- 1 root root  9145 Nov 13 17:11 README_COLAB.md
-rw-r--r-- 1 root root 11062 Nov 13 17:11 RE

## 3️⃣ Pipeline'ı Çalıştır

### Seçenek A: Tüm Pipeline (Tavsiye Edilen)

Tüm adımları otomatik çalıştırır (~1-2 saat)

In [7]:
# Tüm pipeline'ı çalıştır (non-interactive Colab versiyonu)
# Not: Bu adım 1-2 saat sürebilir

print("🚀 Pipeline başlatılıyor...\n")
print("⏱️  Tahmini süre: 1-2 saat")
print("⚠️  Bağlantınızı açık tutun!\n")

# Colab versiyonunu kullan (otomatik, kullanıcı girişi gerektirmeyen)
!python run_pipeline_colab.py

🚀 Pipeline başlatılıyor...

⏱️  Tahmini süre: 1-2 saat
⚠️  Bağlantınızı açık tutun!


KETAMINE DOCKING PIPELINE - GOOGLE COLAB

Started: 2025-11-13 17:11:37
Working directory: /content/comp-chem-scripts/ketamine-docking

Steps to run:
  ✓ Download PDB Structures
  ✓ Prepare Ketamine Ligand
  ✓ Prepare Protein Structures
  ✓ Run AutoDock Vina Docking
  ✓ Analyze Results and Generate Reports

🚀 Starting pipeline automatically (Colab mode - no user input required)
⏱️  Estimated time: 1-2 hours for complete analysis


STEP: Download PDB Structures

Running: 1_download_structures.py
Started: 17:11:37

KETAMINE DOCKING PIPELINE - PDB Structure Downloader


TARGET: NMDA (Priority 1)
Description: NMDA Receptor GluN2B subunit - Primary ketamine target
Structures to download: 3

[7EU8] GluN1-GluN2B + S-ketamine complex (human)
✓ Saved to 7eu8.pdb
[4PE5] GluN1a/GluN2B NMDA receptor
✓ Saved to 4pe5.pdb
[5IPR] GluN1/GluN2B NMDA receptor
✓ Saved to 5ipr.pdb

TARGET: EGFR (Priority 2)
Description: EG

### Seçenek B: Adım Adım Çalıştırma

Her adımı ayrı ayrı çalıştırıp kontrol edebilirsiniz.

In [8]:
# Adım 1: PDB yapılarını indir
print("⬇️ Adım 1: PDB yapıları indiriliyor...\n")
!python scripts/1_download_structures.py

⬇️ Adım 1: PDB yapıları indiriliyor...

KETAMINE DOCKING PIPELINE - PDB Structure Downloader


TARGET: NMDA (Priority 1)
Description: NMDA Receptor GluN2B subunit - Primary ketamine target
Structures to download: 3

[7EU8] GluN1-GluN2B + S-ketamine complex (human)
✓ Saved to 7eu8.pdb
[4PE5] GluN1a/GluN2B NMDA receptor
✓ Saved to 4pe5.pdb
[5IPR] GluN1/GluN2B NMDA receptor
✓ Saved to 5ipr.pdb

TARGET: EGFR (Priority 2)
Description: EGFR kinase domain - EGF expression significantly reduced
Structures to download: 3

[4I24] EGFR kinase domain (wild-type)
✓ Saved to 4i24.pdb
[4G5J] EGFR kinase + inhibitor complex
✓ Saved to 4g5j.pdb
[2GS6] Active EGFR kinase domain
✓ Saved to 2gs6.pdb

TARGET: CSNK1D (Priority 3)
Description: Casein Kinase 1 Delta - Wnt/β-catenin pathway
Structures to download: 2

[6GZM] CSNK1D with inhibitor
✓ Saved to 6gzm.pdb
[5OKS] CSNK1D catalytic domain
✓ Saved to 5oks.pdb

DOWNLOAD SUMMARY
Total structures: 8
Successfully downloaded: 8
Failed: 0

All structures saved

In [9]:
# Adım 2: Ketamin ligandını hazırla
print("🧪 Adım 2: Ketamin ligandı hazırlanıyor...\n")
!python scripts/2_prepare_ligand.py

🧪 Adım 2: Ketamin ligandı hazırlanıyor...

KETAMINE DOCKING PIPELINE - Ligand Preparation


Preparing: S-ketamine
Description: S-enantiomer (Esketamine) - More potent NMDA antagonist
SMILES: C[C@@H]1[C@@H](C(=O)c2ccccc2Cl)CCCCN1C
  Molecular properties:
    MW: 265.78 g/mol
    LogP: 3.64
    H-bond donors: 0
    H-bond acceptors: 2
  ✓ 3D structure saved to S-ketamine.pdb
  ✓ Also saved as MOL2 and SDF formats
  ✓ PDBQT file created: S-ketamine.pdbqt

Preparing: R-ketamine
Description: R-enantiomer - Less potent but still active
SMILES: C[C@H]1[C@H](C(=O)c2ccccc2Cl)CCCCN1C
  Molecular properties:
    MW: 265.78 g/mol
    LogP: 3.64
    H-bond donors: 0
    H-bond acceptors: 2
  ✓ 3D structure saved to R-ketamine.pdb
  ✓ Also saved as MOL2 and SDF formats
  ✓ PDBQT file created: R-ketamine.pdbqt

Preparing: racemic-ketamine
Description: Racemic mixture (no stereochemistry)
SMILES: CC1C(C(=O)c2ccccc2Cl)CCCCN1C
  Molecular properties:
    MW: 265.78 g/mol
    LogP: 3.64
    H-bond donors

In [10]:
# Adım 3: Proteinleri hazırla
print("🔧 Adım 3: Proteinler hazırlanıyor...\n")
!python scripts/3_prepare_proteins.py

🔧 Adım 3: Proteinler hazırlanıyor...

KETAMINE DOCKING PIPELINE - Protein Preparation


Processing: CSNK1D
Found 2 PDB files

  [5OKS]
    Removing water and ions... ✓
    Converting to PDBQT... ✓
  [6GZM]
    Removing water and ions... ✓
    Converting to PDBQT... ✓

Processing: EGFR
Found 3 PDB files

  [2GS6]
    Removing water and ions... ✓
    Converting to PDBQT... ✓
  [4G5J]
    Removing water and ions... ✓
    Converting to PDBQT... ✓
  [4I24]
    Removing water and ions... ✓
    Converting to PDBQT... ✓

Processing: NMDA
Found 3 PDB files

  [4PE5]
    Removing water and ions... ✓
    Converting to PDBQT... ✓
  [5IPR]
    Removing water and ions... ✓
    Converting to PDBQT... ✓
  [7EU8]
    Removing water and ions... ✓
    Converting to PDBQT... ✓

PROTEIN PREPARATION SUMMARY
Total structures processed: 8
Output directory: /content/comp-chem-scripts/ketamine-docking/data/prepared

✓ 8 structures prepared!

Checking docking tools...
  ✓ Open Babel found


In [11]:
# Adım 4: Docking yap (EN UZUN ADIM - 1+ saat)
print("🎯 Adım 4: Docking başlatılıyor...\n")
print("⚠️ Bu adım 1+ saat sürebilir\n")
!python scripts/4_run_docking.py

🎯 Adım 4: Docking başlatılıyor...

⚠️ Bu adım 1+ saat sürebilir

KETAMINE DOCKING PIPELINE - AutoDock Vina Docking
Started: 2025-11-13 17:12:29


TARGET: NMDA
Description: NMDA Receptor GluN2B subunit - Primary ketamine binding site
Priority: 1

Structures to dock: 3

  [7EU8] GluN1-GluN2B + S-ketamine complex (human)

    Docking S-ketamine...
    Running AutoDock Vina...
    ✗ Vina failed with return code 1
    Error: Command line parse error: unrecognised option '--log'

Correct usage:

Input:
  --receptor arg             rigid part of the receptor (PDBQT)
  --flex arg                 flexible side chains, if any (PDBQT)
  --ligand arg               ligand (PDBQT)
  --batch arg                batch ligand (PDBQT)
  --scoring arg (=vina)      scoring function (ad4, vina or vinardo)

Search space (required):
  --maps arg                 affinity maps for the autodock4.2 (ad4) or vina 
                             scoring function
  --center_x arg             X coordinate of the center

In [12]:
# Adım 5: Sonuçları analiz et
print("📊 Adım 5: Sonuçlar analiz ediliyor...\n")
!python scripts/5_analyze_results.py

📊 Adım 5: Sonuçlar analiz ediliyor...

KETAMINE DOCKING PIPELINE - Results Analysis



## 4️⃣ Sonuçları Görüntüle

In [13]:
# Genel özeti göster
import glob

summary_files = glob.glob('data/results/reports/overall_summary_*.txt')
if summary_files:
    latest_summary = sorted(summary_files)[-1]
    print("="*70)
    print("GENEL ÖZET")
    print("="*70)
    with open(latest_summary, 'r') as f:
        print(f.read())
else:
    print("❌ Henüz sonuç bulunamadı. Pipeline'ı çalıştırdınız mı?")

❌ Henüz sonuç bulunamadı. Pipeline'ı çalıştırdınız mı?


In [14]:
# Özet tabloyu göster
import pandas as pd
import glob

csv_files = glob.glob('data/results/reports/summary_*.csv')
if csv_files:
    latest_csv = sorted(csv_files)[-1]
    df = pd.read_csv(latest_csv)
    print("\n📊 Docking Sonuçları Özet Tablosu:\n")
    print(df.to_string(index=False))

    # Grafik çiz
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(12, 6))

    # Her target için en iyi affinity
    best_per_target = df.groupby('Target')['Best Affinity (kcal/mol)'].min()

    colors = ['green' if x <= -7 else 'orange' if x <= -5 else 'red'
              for x in best_per_target.values]

    best_per_target.plot(kind='bar', ax=ax, color=colors)
    ax.set_ylabel('Binding Affinity (kcal/mol)')
    ax.set_xlabel('Target')
    ax.set_title('En İyi Bağlanma Affinitesi (Hedef Bazında)')
    ax.axhline(y=-7, color='green', linestyle='--', label='Güçlü bağlanma')
    ax.axhline(y=-5, color='orange', linestyle='--', label='Zayıf bağlanma')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("❌ CSV sonuç dosyası bulunamadı")

❌ CSV sonuç dosyası bulunamadı


## 5️⃣ Sonuçları İndir

Colab geçici olduğu için sonuçları bilgisayarınıza indirin!

In [15]:
# Sonuçları ZIP olarak hazırla
import shutil
import os
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
zip_name = f'ketamine_docking_results_{timestamp}'

print(f"📦 Sonuçlar paketleniyor: {zip_name}.zip\n")

# ZIP oluştur
shutil.make_archive(zip_name, 'zip', 'data/results')

print(f"✓ Paketleme tamamlandı: {zip_name}.zip")
print(f"Dosya boyutu: {os.path.getsize(f'{zip_name}.zip') / 1024 / 1024:.2f} MB\n")

# İndir
from google.colab import files

print("⬇️ İndirme başlatılıyor...")
files.download(f'{zip_name}.zip')

print("\n✓ İndirme tamamlandı!")
print("\nZIP içeriği:")
print("  - reports/: Tüm raporlar (TXT, Excel, CSV)")
print("  - nmda/: NMDA docking sonuçları")
print("  - egfr/: EGFR docking sonuçları")
print("  - csnk1d/: CSNK1D docking sonuçları")
print("  - all_results.json: Ham veri")

📦 Sonuçlar paketleniyor: ketamine_docking_results_20251113_171233.zip

✓ Paketleme tamamlandı: ketamine_docking_results_20251113_171233.zip
Dosya boyutu: 0.01 MB

⬇️ İndirme başlatılıyor...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✓ İndirme tamamlandı!

ZIP içeriği:
  - reports/: Tüm raporlar (TXT, Excel, CSV)
  - nmda/: NMDA docking sonuçları
  - egfr/: EGFR docking sonuçları
  - csnk1d/: CSNK1D docking sonuçları
  - all_results.json: Ham veri


In [16]:
# Alternatif: Sadece raporları indir (daha küçük)
import glob
import os
from google.colab import files

print("📄 Sadece raporlar indiriliyor...\n")

report_files = glob.glob('data/results/reports/*')

for report_file in report_files:
    if os.path.isfile(report_file):
        print(f"  Downloading: {os.path.basename(report_file)}")
        files.download(report_file)

print("\n✓ Rapor indirme tamamlandı!")

📄 Sadece raporlar indiriliyor...


✓ Rapor indirme tamamlandı!


## 🔧 Hızlı Test (Opsiyonel)

Tüm pipeline'ı çalıştırmadan önce hızlı test yapmak isterseniz:

In [17]:
# HIZLI TEST: Sadece NMDA + tek yapı (7EU8)
# Düşük exhaustiveness ile (~10 dakika)

print("🧪 Hızlı test modu\n")

# 1. Tek yapı indir
!python scripts/1_download_structures.py

# 2. Ligand hazırla
!python scripts/2_prepare_ligand.py

# 3. Proteinleri hazırla
!python scripts/3_prepare_proteins.py

# 4. Config'i düzenle - düşük exhaustiveness
import yaml

with open('configs/nmda_targets.yaml', 'r') as f:
    config = yaml.safe_load(f)

# Sadece 7EU8'i tut, exhaustiveness düşür
config['structures'] = {
    '7EU8': config['structures']['7EU8']
}
config['structures']['7EU8']['exhaustiveness'] = 8
config['structures']['7EU8']['num_modes'] = 5

with open('configs/nmda_targets.yaml', 'w') as f:
    yaml.dump(config, f)

print("\n✓ Test konfigürasyonu hazır")
print("Şimdi docking'i çalıştırın (scripts/4_run_docking.py)")

🧪 Hızlı test modu

KETAMINE DOCKING PIPELINE - PDB Structure Downloader


TARGET: NMDA (Priority 1)
Description: NMDA Receptor GluN2B subunit - Primary ketamine target
Structures to download: 3

[7EU8] GluN1-GluN2B + S-ketamine complex (human)
✓ Saved to 7eu8.pdb
[4PE5] GluN1a/GluN2B NMDA receptor
✓ Saved to 4pe5.pdb
[5IPR] GluN1/GluN2B NMDA receptor
✓ Saved to 5ipr.pdb

TARGET: EGFR (Priority 2)
Description: EGFR kinase domain - EGF expression significantly reduced
Structures to download: 3

[4I24] EGFR kinase domain (wild-type)
✓ Saved to 4i24.pdb
[4G5J] EGFR kinase + inhibitor complex
✓ Saved to 4g5j.pdb
[2GS6] Active EGFR kinase domain
✓ Saved to 2gs6.pdb

TARGET: CSNK1D (Priority 3)
Description: Casein Kinase 1 Delta - Wnt/β-catenin pathway
Structures to download: 2

[6GZM] CSNK1D with inhibitor
✓ Saved to 6gzm.pdb
[5OKS] CSNK1D catalytic domain
✓ Saved to 5oks.pdb

DOWNLOAD SUMMARY
Total structures: 8
Successfully downloaded: 8
Failed: 0

All structures saved to: /content/comp-ch

## 💡 İpuçları

1. **Runtime Süresi**: Colab ücretsiz max 12 saat. Pipeline 1-2 saatte biter.

2. **Sonuçları Hemen İndirin**: Colab geçici storage kullanır, session kapanınca silinir.

3. **Yeniden Başlatma**: Eğer bağlantı koptu:
   - Sonuçlar kaybolabilir
   - Eğer docking tamamlanmışsa, sadece analiz adımını çalıştırın

4. **Hız Artırma**:
   - `exhaustiveness` değerini düşürün (24 → 16)
   - Daha az yapı kullanın (her hedeften 1 tane)

5. **GPU**: Bu pipeline CPU tabanlı, GPU gerekmez.

## 📚 Dokümantasyon

- [README.md](https://github.com/egundeger/comp-chem-scripts/blob/main/ketamine-docking/README.md) - Kapsamlı dokümantasyon
- [QUICKSTART.md](https://github.com/egundeger/comp-chem-scripts/blob/main/ketamine-docking/QUICKSTART.md) - Hızlı başlangıç
- [README_COLAB.md](https://github.com/egundeger/comp-chem-scripts/blob/main/ketamine-docking/README_COLAB.md) - Colab kılavuzu

## ❓ Sorun Giderme

**"Vina not found" hatası:**
```python
!which vina
# Eğer bulunamazsa, Vina kurulum hücresini tekrar çalıştırın
```

**"Module not found" hatası:**
```python
!pip install <eksik_paket>
```

**"ketamine-docking directory not found":**
```python
# Branch ismini kontrol edin, doğru branch'i clone edin
!git clone -b claude/ketamine-docking-pipeline-011CV5ZKC6fFJBRhdCG2Mtt6 https://github.com/egundeger/comp-chem-scripts.git
```

**Docking çok uzun sürüyor:**
- Config dosyalarında `exhaustiveness` değerini düşürün
- Daha az yapı kullanın

**Runtime disconnect:**
- Sonuçları sık sık indirin
- Colab Pro kullanarak daha uzun runtime alabilirsiniz